# 04 - Which parameters are worth calibrating?

ADM1 has dozens of parameters. Fitting all of them is not just slow, it is
meaningless. If a parameter does not change the output you measure, the
optimiser will assign it any value at all.

Two questions decide what goes into a calibration:

1. **Sensitivity** - does the parameter move the measured channel?
2. **Identifiability** - can it be told apart from the others, or do two
   parameters compensate for each other?


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from demo_plant import build_demo_plant, make_twin_measurements, simulate

from pyadm1ode_calibration.calibration.analysis.sensitivity import SensitivityAnalyzer

measurements = make_twin_measurements(days=3, noise=0.02, seed=0)

CANDIDATES = {
    "k_hyd_ch": 4.0,    # hydrolysis of carbohydrates
    "k_hyd_pr": 10.0,   # hydrolysis of proteins
    "k_hyd_li": 10.0,   # hydrolysis of lipids
    "k_dis": 0.5,       # disintegration of composite material
}
print("candidates:", CANDIDATES)

## The direct check first

Before using any analysis class, convince yourself by hand. Vary one parameter
over a wide range and watch the output. If nothing moves, nothing will make it
move.

In [ ]:
plant = build_demo_plant(days=3)

for name, base in CANDIDATES.items():
    outputs = [
        float(np.mean(simulate(plant, measurements, {name: base * factor})["Q_gas"]))
        for factor in (0.25, 1.0, 4.0)
    ]
    spread = (max(outputs) - min(outputs)) / np.mean(outputs)
    print(f"  {name:9s} Q_gas over a 16-fold parameter range: {spread:6.2%}")

`k_dis` does not move the result at all and that is not a bug. PyADM1ODE
characterises substrates directly into carbohydrates, proteins and lipids, so
the disintegration step that `k_dis` governs is bypassed for this feed.
Calibrating it would produce a number with no meaning behind it.

## The same question through `SensitivityAnalyzer`

The analyzer does this systematically. It perturbs each parameter, re-simulates
and reports the local gradient plus a normalised sensitivity per objective.

In [ ]:
analyzer = SensitivityAnalyzer(build_demo_plant(days=3), verbose=False)
results = analyzer.analyze(CANDIDATES, measurements, objectives=["Q_gas"], perturbation=0.2)

rows = sorted(
    ((name, r.normalized_sensitivity["Q_gas"], r.local_gradient["Q_gas"]) for name, r in results.items()),
    key=lambda row: abs(row[1]),
    reverse=True,
)
for name, normalized, gradient in rows:
    print(f"  {name:9s} normalised = {normalized:7.4f}   gradient = {gradient:9.3f}")

In [ ]:
names = [row[0] for row in rows]
values = [abs(row[1]) for row in rows]

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(names[::-1], values[::-1], color="tab:blue")
ax.set_xlabel("normalised sensitivity of Q_gas")
ax.set_title("What is worth calibrating against Q_gas")
fig.tight_layout()